# FakeGuard — AI Fake News Detector
### NeuroLogic '26 Datathon

> Run cells one at a time. Each cell does one job only.
> If training fails after an epoch, do NOT restart from scratch — use the resume cell (8B).

### Dataset
- **Source:** https://www.kaggle.com/datasets/clmentbisaillon/fake-and-real-news-dataset
- **Files:** Fake.csv (23,481 fake articles) + True.csv (21,417 real articles)
- **Columns in both files:** title, text, subject, date
- **Labels assigned by pipeline:** Fake.csv → FALSE (0) | True.csv → TRUE (1)
- **Add to notebook:** Notebook → Add Data → search 'clmentbisaillon fake and real news dataset'
- **Kaggle path:** `/kaggle/input/fake-and-real-news-dataset/`

### Beginner safety notes
- Use **GPU T4**, not P100.
- Run **Cell 1 and Cell 2 only once** at the very beginning.
- After a kernel restart, start from **Cell 3**.
- Cell 8 is self-contained and does not import src/train.py.
- If training crashes mid-way, use Cell 8B to resume from last checkpoint.

### Notebook flow
| Cell | Task | Time |
|---|---|---|
| 0 | Inspect dataset files | instant |
| 1 | Fix packages | 30 sec |
| 2 | Restart kernel | instant |
| 3 | GPU check | instant |
| 4 | Clone/pull repo | 30 sec |
| 5 | Locate Fake.csv + True.csv | instant |
| 6 | Preprocess data (merge + clean + split) | 2 min |
| 7 | Baseline model + XAI feature importance | 2 min |
| 8 | Train RoBERTa | ~30 min |
| 8B | Resume training (only if needed) | — |
| 9 | Evaluate model | 3 min |
| 10 | Generate predictions.csv | 2 min |
| 11 | Launch Gradio demo for judges | 1 min |


## Cell 0 — Inspect Dataset Files ⚠️ RUN THIS FIRST
Confirms Fake.csv and True.csv are available. Shows columns + sample rows.
If you see both files, you are ready to proceed.

In [ ]:
import pandas as pd, glob
print('=== Scanning /kaggle/input for CSV files ===')
matches = sorted(glob.glob('/kaggle/input/**/*.csv', recursive=True))
if not matches:
    print('❌ No CSV files found.')
    print('   Add the dataset: Notebook → Add Data → search clmentbisaillon fake and real news dataset')
else:
    for path in matches:
        try:
            df_peek = pd.read_csv(path, nrows=3)
            print(f'\n📂 {path}')
            print(f'   Columns : {list(df_peek.columns)}')
            print(f'   Shape   : {df_peek.shape}')
            print(df_peek.head(2).to_string())
        except Exception as e:
            print(f'Could not read {path}: {e}')
print('\n=== You should see Fake.csv and True.csv above ===')
print('=== Both must have columns: title, text, subject, date ===')

## Cell 1 — Fix Package Conflicts

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'uninstall', '-y', 'peft'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers', 'accelerate', 'datasets', 'seaborn',
])
print('✅ Done. Now run Cell 2.')

## Cell 2 — Restart Kernel
Run once. After restart, skip Cells 1 and 2 and begin from Cell 3.

In [ ]:
import IPython
print('Restarting kernel... after restart begin from Cell 3.')
IPython.Application.instance().kernel.do_shutdown(restart=True)

## Cell 3 — GPU Check

In [ ]:
import torch
print(f'PyTorch : {torch.__version__}')
if not torch.cuda.is_available():
    raise RuntimeError('No GPU. Go to Settings → Accelerator → GPU T4 x1')
gpu_name = torch.cuda.get_device_name(0)
print(f'GPU     : {gpu_name}')
print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
if 'P100' in gpu_name:
    raise RuntimeError('Switch to T4 in notebook settings — P100 is not supported.')
print('✅ GPU ready.')

## Cell 4 — Clone or Pull Repo

In [ ]:
import subprocess, os, sys
REPO_URL  = 'https://github.com/ayushtiwari18/neurologic-datathon-fakenews.git'
REPO_ROOT = '/kaggle/working/neurologic-datathon-fakenews'
if not os.path.exists(REPO_ROOT):
    subprocess.check_call(['git', 'clone', REPO_URL, REPO_ROOT])
else:
    subprocess.check_call(['git', '-C', REPO_ROOT, 'pull'])
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)
for d in ['data/raw', 'data/processed', 'outputs', 'models/roberta_fakenews']:
    os.makedirs(f'{REPO_ROOT}/{d}', exist_ok=True)
print(f'✅ Repo ready: {os.getcwd()}')

## Cell 5 — Locate Fake.csv and True.csv
Finds both dataset files and stores their exact paths for Cell 6.

In [ ]:
import glob
from pathlib import Path

REPO_ROOT     = '/kaggle/working/neurologic-datathon-fakenews'
PROCESSED_DIR = Path(f'{REPO_ROOT}/data/processed')
MODEL_SAVE    = Path(f'{REPO_ROOT}/models/roberta_fakenews')
OUTPUTS_DIR   = Path(f'{REPO_ROOT}/outputs')

all_csvs = sorted(glob.glob('/kaggle/input/**/*.csv', recursive=True))

fake_matches = [p for p in all_csvs if Path(p).name.lower() == 'fake.csv']
true_matches = [p for p in all_csvs if Path(p).name.lower() == 'true.csv']

if not fake_matches:
    raise FileNotFoundError('❌ Fake.csv not found. Add: clmentbisaillon/fake-and-real-news-dataset')
if not true_matches:
    raise FileNotFoundError('❌ True.csv not found. Add: clmentbisaillon/fake-and-real-news-dataset')

FAKE_CSV = fake_matches[0]
TRUE_CSV = true_matches[0]

print(f'✅ Fake.csv : {FAKE_CSV}')
print(f'✅ True.csv : {TRUE_CSV}')
print(f'✅ Paths stored as FAKE_CSV and TRUE_CSV. Ready for Cell 6.')

## Cell 6 — Preprocess Data
Calls `src/preprocess.py` with the exact paths to Fake.csv and True.csv.
The pipeline handles everything:
1. Assigns FALSE label to Fake.csv rows, TRUE label to True.csv rows
2. Merges + shuffles (44,898 total rows)
3. Cleans text (lowercase, strip URLs/HTML/special chars)
4. Drops the `date` column (noise)
5. Builds `combined` = subject + [SEP] + title + [SEP] + text
6. Encodes labels: TRUE→1, FALSE→0
7. Stratified 70/15/15 split
8. Saves train.csv, val.csv, test.csv to data/processed/

In [ ]:
import sys, os, pandas as pd
from pathlib import Path

REPO_ROOT     = '/kaggle/working/neurologic-datathon-fakenews'
PROCESSED_DIR = Path(f'{REPO_ROOT}/data/processed')

if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)

# Check if already preprocessed (skip if re-running after a crash)
all_exist = all((PROCESSED_DIR / f).exists() for f in ['train.csv', 'val.csv', 'test.csv'])
if all_exist:
    train_df = pd.read_csv(PROCESSED_DIR / 'train.csv')
    val_df   = pd.read_csv(PROCESSED_DIR / 'val.csv')
    test_df  = pd.read_csv(PROCESSED_DIR / 'test.csv')
    print('✅ Processed CSVs already exist — loaded directly. Skipping re-preprocessing.')
else:
    # Require Cell 5 to have run
    if 'FAKE_CSV' not in dir() or 'TRUE_CSV' not in dir():
        raise RuntimeError('Run Cell 5 first to locate Fake.csv and True.csv.')
    from src.preprocess import run_preprocessing
    train_df, val_df, test_df = run_preprocessing(
        fake_csv_path=FAKE_CSV,
        true_csv_path=TRUE_CSV,
        processed_dir=str(PROCESSED_DIR),
    )

print(f'\nTrain : {len(train_df):,} rows')
print(f'Val   : {len(val_df):,} rows')
print(f'Test  : {len(test_df):,} rows')
print(f'Columns: {list(train_df.columns)}')
print(f'\nLabel distribution in train:')
print(train_df['label'].value_counts().to_string())

## Cell 7 — Baseline Model + Feature Importance XAI
TF-IDF + Logistic Regression baseline.
Also prints top 20 words for FALSE (fake) and TRUE (real) — this is Explainable AI (XAI) at the baseline level.

In [ ]:
import pandas as pd, numpy as np
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

REPO_ROOT     = '/kaggle/working/neurologic-datathon-fakenews'
PROCESSED_DIR = Path(f'{REPO_ROOT}/data/processed')

train_df = pd.read_csv(PROCESSED_DIR / 'train.csv')
val_df   = pd.read_csv(PROCESSED_DIR / 'val.csv')

vectorizer = TfidfVectorizer(max_features=50000, ngram_range=(1, 2))
X_train = vectorizer.fit_transform(train_df['combined'])
X_val   = vectorizer.transform(val_df['combined'])

lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, train_df['label'])

baseline_preds = lr.predict(X_val)
baseline_acc   = accuracy_score(val_df['label'], baseline_preds)
print(f'Baseline (TF-IDF + LR) Accuracy : {baseline_acc:.4f}')
print(classification_report(val_df['label'], baseline_preds, target_names=['FALSE (fake)', 'TRUE (real)']))

print('\n' + '='*60)
print('XAI — Baseline Feature Importance')
print('='*60)
feature_names = np.array(vectorizer.get_feature_names_out())
coefficients  = lr.coef_[0]
fake_idx  = np.argsort(coefficients)[:20]
real_idx  = np.argsort(coefficients)[-20:][::-1]
print('\n🔴 Top 20 words → FALSE (fake news):')
for w, c in zip(feature_names[fake_idx], coefficients[fake_idx]): print(f'  {w:<30} {c:>10.4f}')
print('\n🟢 Top 20 words → TRUE (real news):')
for w, c in zip(feature_names[real_idx], coefficients[real_idx]): print(f'  {w:<30} {c:>10.4f}')

## Cell 8 — Train RoBERTa
Self-contained. Does NOT import src/train.py.
Labels: FALSE=0 (fake), TRUE=1 (real). Targets ~98%+ validation accuracy.

In [ ]:
import os, json, torch, numpy as np, pandas as pd
from pathlib import Path
from torch.utils.data import Dataset
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoTokenizer, RobertaForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding, EarlyStoppingCallback,
)

REPO_ROOT     = '/kaggle/working/neurologic-datathon-fakenews'
PROCESSED_DIR = Path(f'{REPO_ROOT}/data/processed')
MODEL_SAVE    = Path(f'{REPO_ROOT}/models/roberta_fakenews')
OUTPUTS_DIR   = Path(f'{REPO_ROOT}/outputs')
MODEL_SAVE.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

# ── Dataset class ───────────────────────────────────────────────────────
class FakeNewsDataset(Dataset):
    def __init__(self, csv_path, tokenizer, max_length=512):
        self.df = pd.read_csv(csv_path)
        self.tokenizer = tokenizer
        self.max_length = max_length
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        enc = self.tokenizer(
            str(row['combined']),
            truncation=True,
            max_length=self.max_length,
            padding=False,
        )
        return {
            'input_ids'     : enc['input_ids'],
            'attention_mask': enc['attention_mask'],
            'labels'        : int(row['label']),  # 0=FALSE/fake, 1=TRUE/real
        }

# ── Metrics ─────────────────────────────────────────────────────────────
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1'      : f1_score(labels, preds, average='weighted'),
    }

# ── Load model ──────────────────────────────────────────────────────────
print('Loading tokenizer and model...')
tokenizer = AutoTokenizer.from_pretrained('roberta-base')
model = RobertaForSequenceClassification.from_pretrained(
    'roberta-base',
    num_labels=2,
    id2label={0: 'FALSE', 1: 'TRUE'},
    label2id={'FALSE': 0, 'TRUE': 1},
)

# ── Datasets ────────────────────────────────────────────────────────────
train_dataset = FakeNewsDataset(PROCESSED_DIR / 'train.csv', tokenizer)
val_dataset   = FakeNewsDataset(PROCESSED_DIR / 'val.csv',   tokenizer)
print(f'Train: {len(train_dataset):,} samples | Val: {len(val_dataset):,} samples')

# ── Training args ───────────────────────────────────────────────────────
training_args = TrainingArguments(
    output_dir=str(MODEL_SAVE),
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_steps=500,
    weight_decay=0.01,
    learning_rate=2e-5,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    report_to='none',
    logging_steps=100,
    seed=42,
    save_total_limit=2,
)

# ── Trainer ─────────────────────────────────────────────────────────────
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

# ── Train ───────────────────────────────────────────────────────────────
print('\n' + '='*60)
print('STARTING ROBERTA TRAINING')
print('Expected time: ~30 minutes on T4 GPU')
print('='*60)
train_result = trainer.train()
eval_metrics = trainer.evaluate()

# ── Save ────────────────────────────────────────────────────────────────
trainer.save_model(str(MODEL_SAVE))
tokenizer.save_pretrained(str(MODEL_SAVE))

metrics_out = {
    'final_val_accuracy': round(eval_metrics['eval_accuracy'], 6),
    'final_val_f1'      : round(eval_metrics['eval_f1'], 6),
    'train_runtime_sec' : round(train_result.metrics.get('train_runtime', 0), 2),
}
json.dump(metrics_out, open(OUTPUTS_DIR / 'training_metrics.json', 'w'), indent=2)

print(f'\n✅ Val Accuracy : {eval_metrics["eval_accuracy"]:.4f}')
print(f'✅ Val F1       : {eval_metrics["eval_f1"]:.4f}')
print('✅ Model saved to:', MODEL_SAVE)

## Cell 8B — Resume Training From Checkpoint (Only If Needed)
Use ONLY if Cell 8 trained at least 1 epoch then crashed. Picks up from last saved checkpoint.

In [ ]:
import os, json, torch, numpy as np, pandas as pd
from pathlib import Path
from torch.utils.data import Dataset
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoTokenizer, RobertaForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding, EarlyStoppingCallback,
)

REPO_ROOT     = '/kaggle/working/neurologic-datathon-fakenews'
PROCESSED_DIR = Path(f'{REPO_ROOT}/data/processed')
MODEL_SAVE    = Path(f'{REPO_ROOT}/models/roberta_fakenews')
OUTPUTS_DIR   = Path(f'{REPO_ROOT}/outputs')

checkpoints = sorted(MODEL_SAVE.glob('checkpoint-*'), key=os.path.getmtime)
if not checkpoints:
    raise FileNotFoundError('No checkpoint found. Run Cell 8 for a fresh start.')
latest_ckpt = str(checkpoints[-1])
print(f'Resuming from: {latest_ckpt}')

class FakeNewsDataset(Dataset):
    def __init__(self, csv_path, tokenizer, max_length=512):
        self.df = pd.read_csv(csv_path)
        self.tokenizer = tokenizer
        self.max_length = max_length
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        enc = self.tokenizer(str(row['combined']), truncation=True, max_length=self.max_length, padding=False)
        return {'input_ids': enc['input_ids'], 'attention_mask': enc['attention_mask'], 'labels': int(row['label'])}

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {'accuracy': accuracy_score(labels, preds), 'f1': f1_score(labels, preds, average='weighted')}

tokenizer = AutoTokenizer.from_pretrained('roberta-base')
model = RobertaForSequenceClassification.from_pretrained(
    latest_ckpt, num_labels=2,
    id2label={0: 'FALSE', 1: 'TRUE'},
    label2id={'FALSE': 0, 'TRUE': 1},
)

train_dataset = FakeNewsDataset(PROCESSED_DIR / 'train.csv', tokenizer)
val_dataset   = FakeNewsDataset(PROCESSED_DIR / 'val.csv',   tokenizer)

training_args = TrainingArguments(
    output_dir=str(MODEL_SAVE),
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_steps=500,
    weight_decay=0.01,
    learning_rate=2e-5,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    report_to='none',
    logging_steps=100,
    seed=42,
    save_total_limit=2,
)

trainer = Trainer(
    model=model, args=training_args,
    train_dataset=train_dataset, eval_dataset=val_dataset,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

train_result = trainer.train(resume_from_checkpoint=latest_ckpt)
eval_metrics = trainer.evaluate()
trainer.save_model(str(MODEL_SAVE))
tokenizer.save_pretrained(str(MODEL_SAVE))

metrics_out = {
    'final_val_accuracy': round(eval_metrics['eval_accuracy'], 6),
    'final_val_f1'      : round(eval_metrics['eval_f1'], 6),
    'train_runtime_sec' : round(train_result.metrics.get('train_runtime', 0), 2),
    'resumed_from'      : latest_ckpt,
}
json.dump(metrics_out, open(OUTPUTS_DIR / 'training_metrics.json', 'w'), indent=2)
print(f'✅ Val Accuracy : {eval_metrics["eval_accuracy"]:.4f}')
print(f'✅ Val F1       : {eval_metrics["eval_f1"]:.4f}')
print('✅ Resume complete.')

## Cell 9 — Evaluate Model

In [ ]:
import sys, os
from pathlib import Path
from IPython.display import Image, display
REPO_ROOT     = '/kaggle/working/neurologic-datathon-fakenews'
PROCESSED_DIR = Path(f'{REPO_ROOT}/data/processed')
if REPO_ROOT not in sys.path: sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)
from src.evaluate import run_evaluation
try:
    baseline_acc
except NameError:
    baseline_acc = 0.9466
eval_report = run_evaluation(val_csv_path=str(PROCESSED_DIR / 'val.csv'), baseline_accuracy=baseline_acc)
cm_path = f'{REPO_ROOT}/outputs/confusion_matrix.png'
if os.path.exists(cm_path): display(Image(cm_path))
print(f"RoBERTa Accuracy  : {eval_report['metrics']['accuracy']:.4f}")
print(f"Baseline Accuracy : {baseline_acc:.4f}")
print(f"Improvement       : +{eval_report['metrics']['accuracy'] - baseline_acc:.4f}")

## Cell 10 — Generate predictions.csv

In [ ]:
import sys, os
from pathlib import Path
REPO_ROOT     = '/kaggle/working/neurologic-datathon-fakenews'
PROCESSED_DIR = Path(f'{REPO_ROOT}/data/processed')
if REPO_ROOT not in sys.path: sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)
from src.predict import run_predict
predictions_df = run_predict(test_csv_path=str(PROCESSED_DIR / 'test.csv'))
print(f'Total     : {len(predictions_df):,}')
print(f'FALSE (0) : {(predictions_df["predicted"] == 0).sum():,}')
print(f'TRUE  (1) : {(predictions_df["predicted"] == 1).sum():,}')
print('✅ Saved to outputs/predictions.csv')

## Cell 11 — Launch Gradio Demo + XAI Suite
Launches interactive demo with a public URL for judges.
Keep this cell running — stopping it kills the public link.

In [ ]:
import subprocess, sys, os
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'gradio>=4.0.0', 'lime'])
REPO_ROOT = '/kaggle/working/neurologic-datathon-fakenews'
if REPO_ROOT not in sys.path: sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)
from app.gradio_demo import launch_demo
print('=' * 60)
print('Launching FakeGuard — XAI Fake News Detector')
print('Public link appears below in ~30 seconds.')
print('='*60)
launch_demo(share=True)